# 11 - Expanded-data arrival-delay models and prediction

This notebook continues the earlier model flow using notebook 10 Parquet.
It compares the historical baseline, Ridge, Random Forest, Gradient-Boosted
Trees, XGBoost and CatBoost under one temporal contract.

Model selection uses December 2022 only. March and June 2023 remain locked
until a winner is frozen. Every model is measured with MAE, RMSE, median
absolute error and p90 absolute error, including punctual and delayed segments.

In [ ]:
from pathlib import Path
import json
import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import Ridge

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.t60_modeling import (
    HistoricalMedianBaseline, MixedCategoricalRidgePreprocessor,
    add_schedule_features, compact_score, segment_metrics,
)

DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "expanded_arrival_pre_t60"
REPORT_ROOT = PROJECT_ROOT / "reports" / "expanded_models"
MODEL_ROOT = PROJECT_ROOT / "models" / "expanded"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
TARGET = "Arrival_Delay_Min"
RUN_MODELS = False
TRAIN_SAMPLE_PERCENT = 10
VALIDATION_SAMPLE_PERCENT = 5
TREE_SAMPLE_PERCENT = 1
SEED = 42

## 1. Load frozen splits and audit leakage

Notebook 10 must first write Parquet. Sample percentages control resources,
not dates. Test labels are loaded for final scoring only after validation
freezes the winner.

In [ ]:
def deterministic_sample(frame, percent):
    if percent >= 100:
        return frame.copy()
    bucket = pd.util.hash_pandas_object(frame["ECTRL ID"], index=False) % 100
    return frame.loc[bucket < percent].copy()

if RUN_MODELS:
    development_splits = {
        name: add_schedule_features(pd.read_parquet(DATA_ROOT / name))
        for name in ("train", "validation")
    }
    forbidden = {"ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
                 "Departure_Delay_Min", "Actual Distance Flown (nm)"}
    assert all(not (forbidden & set(frame.columns)) for frame in development_splits.values())
    cutoff = (pd.to_datetime(development_splits["train"]["FILED OFF BLOCK TIME"])
              - pd.to_datetime(development_splits["train"]["prediction_cutoff_t60"])
             ).dt.total_seconds()/60
    assert np.allclose(cutoff, 60)
    train = deterministic_sample(development_splits["train"], TRAIN_SAMPLE_PERCENT)
    validation = deterministic_sample(
        development_splits["validation"], VALIDATION_SAMPLE_PERCENT
    )
    print({
        **{name: len(frame) for name, frame in development_splits.items()},
        "locked_test_partitions_read": False,
    })
else:
    print("Set RUN_MODELS=True after notebook 10 writes Parquet.")

## 2. Shared features and historical baseline

Airports, operator and grouped aircraft are hashed for Ridge and native
categories for CatBoost. Numeric medians and scaling are learned on train.
Operational T-60 columns are included automatically if notebook 10 built them.

The baseline fallback is route+airline, route, departure-airport+airline,
departure airport and global median, all fitted on train.

In [ ]:
LOW_CARDINALITY_COLUMNS = [
    "STATFOR Market Segment", "Class_aircraft", "Number+Engine Type_aircraft",
]
HIGH_CARDINALITY_COLUMNS = [
    "ADEP", "ADES", "AC Operator", "AC Type_grouped", "AC Registration",
]
CATEGORICAL_COLUMNS = LOW_CARDINALITY_COLUMNS + HIGH_CARDINALITY_COLUMNS
STATIC_NUMERIC_COLUMNS = [
    "Requested_FL_Imputed", "scheduled_duration_min",
    "departure_hour_sin", "departure_hour_cos",
    "departure_dow_sin", "departure_dow_cos", "departure_month",
]
if RUN_MODELS:
    OPERATIONAL_COLUMNS = [
        column for column in train.columns
        if column.startswith(("adep_dep_", "ades_arr_", "route_arr_",
                              "operator_dep_", "operator_arr_", "rotation_"))
    ]
    NUMERIC_COLUMNS = STATIC_NUMERIC_COLUMNS + OPERATIONAL_COLUMNS
    baseline = HistoricalMedianBaseline().fit(train)
    validation_predictions = {
        "historical_baseline": baseline.predict(validation)
    }
    print({"numeric_features": len(NUMERIC_COLUMNS),
           "operational_features": len(OPERATIONAL_COLUMNS)})

## 3. Ridge selection

Ridge remains the primary balanced model: it is memory-efficient, stable with
high-cardinality hashing and was comparatively strong for delayed flights.
Alpha is selected on validation only.

In [ ]:
if RUN_MODELS:
    ridge_rows, ridge_objects = [], {}
    for alpha in (0.1, 1.0, 10.0, 100.0):
        preprocessor = MixedCategoricalRidgePreprocessor(
            LOW_CARDINALITY_COLUMNS, HIGH_CARDINALITY_COLUMNS, NUMERIC_COLUMNS
        )
        x_train = preprocessor.fit_transform(train)
        x_validation = preprocessor.transform(validation)
        model = Ridge(alpha=alpha, solver="lsqr").fit(
            x_train, train[TARGET]
        )
        prediction = model.predict(x_validation)
        metrics = segment_metrics(
            validation[TARGET], prediction, f"ridge_{alpha:g}", "validation"
        )
        ridge_rows.append({"alpha": alpha, **compact_score(metrics)})
        ridge_objects[alpha] = (model, preprocessor, prediction)
    ridge_selection = pd.DataFrame(ridge_rows).sort_values(
        ["combined_MAE_score", "global_MAE"]
    )
    selected_alpha = float(ridge_selection.iloc[0]["alpha"])
    ridge_model, ridge_preprocessor, validation_predictions["ridge"] = (
        ridge_objects[selected_alpha]
    )
    display(ridge_selection)

## 4. Random Forest, GBT and XGBoost

Tree models use a smaller deterministic train sample by default because they
need more RAM, but use identical validation rows. XGBoost is the agreed fourth
model. LightGBM/SynapseML remains a future option with extra runtime complexity.

In [ ]:
def dense_tree_frame(frame, medians=None):
    categorical = frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    hashed = FeatureHasher(
        n_features=512, input_type="string", alternate_sign=False
    ).transform(
        ([f"{column}={value}" for column, value in zip(CATEGORICAL_COLUMNS, row)]
         for row in categorical.itertuples(index=False, name=None))
    ).toarray()
    numeric = frame[NUMERIC_COLUMNS]
    medians = numeric.median() if medians is None else medians
    numeric = numeric.fillna(medians).to_numpy(dtype=np.float32)
    return np.hstack([hashed.astype(np.float32), numeric]), medians

if RUN_MODELS:
    tree_train = deterministic_sample(train, TREE_SAMPLE_PERCENT)
    x_tree_train, tree_medians = dense_tree_frame(tree_train)
    x_tree_validation, _ = dense_tree_frame(validation, tree_medians)
    tree_models = {
        "random_forest": RandomForestRegressor(
            n_estimators=100, max_depth=12, min_samples_leaf=5,
            n_jobs=2, random_state=SEED,
        ),
        "gradient_boosted_trees": HistGradientBoostingRegressor(
            max_iter=200, max_leaf_nodes=31, learning_rate=0.05,
            l2_regularization=5, random_state=SEED,
        ),
    }
    try:
        from xgboost import XGBRegressor
        tree_models["xgboost"] = XGBRegressor(
            n_estimators=400, max_depth=7, learning_rate=0.04,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=8,
            objective="reg:absoluteerror", n_jobs=2, random_state=SEED,
        )
    except ImportError:
        print("XGBoost unavailable; install project requirements.")
    for name, model in tree_models.items():
        model.fit(x_tree_train, tree_train[TARGET])
        validation_predictions[name] = model.predict(x_tree_validation)

## 5. CatBoost

CatBoost is the nonlinear categorical challenger. Native categories capture
route/operator/aircraft interactions without huge one-hot matrices.
Early stopping and time-aware fitting limit overfitting.

In [ ]:
if RUN_MODELS:
    try:
        from catboost import CatBoostRegressor, Pool
        cat_train = train.sort_values("FILED OFF BLOCK TIME").copy()
        cat_validation = validation.sort_values("FILED OFF BLOCK TIME").copy()
        cat_medians = cat_train[NUMERIC_COLUMNS].median()
        for frame in (cat_train, cat_validation):
            frame[CATEGORICAL_COLUMNS] = (
                frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
            )
            frame[NUMERIC_COLUMNS] = frame[NUMERIC_COLUMNS].fillna(cat_medians)
        cat_features = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS
        train_pool = Pool(cat_train[cat_features], cat_train[TARGET],
                          cat_features=CATEGORICAL_COLUMNS)
        validation_pool = Pool(cat_validation[cat_features], cat_validation[TARGET],
                               cat_features=CATEGORICAL_COLUMNS)
        catboost_model = CatBoostRegressor(
            loss_function="MAE", eval_metric="MAE", iterations=1200,
            learning_rate=0.03, depth=7, l2_leaf_reg=8, has_time=True,
            one_hot_max_size=10, max_ctr_complexity=2, random_seed=SEED,
            thread_count=2, od_type="Iter", od_wait=80,
            allow_writing_files=False, verbose=False,
        )
        catboost_model.fit(
            train_pool, eval_set=validation_pool, use_best_model=True
        )
        validation_predictions["catboost"] = catboost_model.predict(
            validation_pool
        )
    except ImportError:
        print("CatBoost unavailable; install project requirements.")

## 6. Freeze winner on validation

Ranking gives equal weight to global MAE and MAE among flights delayed over 15
minutes. RMSE, median and p90 errors remain in the detailed segment report.
No test label is accessed here.

In [ ]:
if RUN_MODELS:
    validation_metrics = pd.concat([
        segment_metrics(validation[TARGET], prediction, name, "validation")
        for name, prediction in validation_predictions.items()
    ], ignore_index=True)
    validation_summary = pd.DataFrame([
        {"candidate": name, **compact_score(group)}
        for name, group in validation_metrics.groupby("model")
    ]).sort_values(["combined_MAE_score", "global_MAE"])
    validation_metrics.to_csv(
        REPORT_ROOT / "validation_segment_metrics.csv", index=False
    )
    validation_summary.to_csv(
        REPORT_ROOT / "validation_model_ranking.csv", index=False
    )
    SELECTED_MODEL = validation_summary.iloc[0]["candidate"]
    (REPORT_ROOT / "selection.json").write_text(
        json.dumps({"selected_model": SELECTED_MODEL}, indent=2),
        encoding="utf-8",
    )
    display(validation_summary)

## 7. Locked March and June evaluation

Run only after validation freezes the winner. March and June are reported
separately to expose temporal drift. A disappointing test must not be folded
back into training and retuned.

In [ ]:
def predict_frozen(name, frame):
    if name == "historical_baseline":
        return baseline.predict(frame)
    if name == "ridge":
        return ridge_model.predict(ridge_preprocessor.transform(frame))
    if name in tree_models:
        matrix, _ = dense_tree_frame(frame, tree_medians)
        return tree_models[name].predict(matrix)
    if name == "catboost":
        prepared = frame.copy()
        prepared[CATEGORICAL_COLUMNS] = (
            prepared[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
        )
        prepared[NUMERIC_COLUMNS] = prepared[NUMERIC_COLUMNS].fillna(cat_medians)
        return catboost_model.predict(
            prepared[CATEGORICAL_COLUMNS + NUMERIC_COLUMNS]
        )
    raise KeyError(name)

if RUN_MODELS:
    locked_splits = {
        name: add_schedule_features(pd.read_parquet(DATA_ROOT / name))
        for name in ("test", "future_test")
    }
    assert all(not (forbidden & set(frame.columns)) for frame in locked_splits.values())
    final_metrics = pd.concat([
        segment_metrics(
            locked_splits[split_name][TARGET],
            predict_frozen(SELECTED_MODEL, locked_splits[split_name]),
            SELECTED_MODEL, split_name,
        )
        for split_name in ("test", "future_test")
    ], ignore_index=True)
    final_metrics.to_csv(REPORT_ROOT / "locked_test_metrics.csv", index=False)
    display(final_metrics)

## 8. Save prediction contract

The bundle records the T-60 horizon, features, temporal periods and sample
fractions so later predictions cannot silently use a different contract.

In [ ]:
if RUN_MODELS:
    bundle = {
        "selected_model_name": SELECTED_MODEL,
        "prediction_horizon_minutes": 60,
        "target": TARGET,
        "categorical_columns": CATEGORICAL_COLUMNS,
        "numeric_columns": NUMERIC_COLUMNS,
        "train_sample_percent": TRAIN_SAMPLE_PERCENT,
        "tree_sample_percent": TREE_SAMPLE_PERCENT,
        "validation_period": "2022-12",
        "test_period": "2023-03",
        "future_test_period": "2023-06",
    }
    if SELECTED_MODEL == "ridge":
        bundle.update({"model": ridge_model, "preprocessor": ridge_preprocessor})
    elif SELECTED_MODEL == "historical_baseline":
        bundle["model"] = baseline
    elif SELECTED_MODEL == "catboost":
        bundle.update({"model": catboost_model, "numeric_medians": cat_medians})
    else:
        bundle.update({"model": tree_models[SELECTED_MODEL],
                       "numeric_medians": tree_medians})
    path = MODEL_ROOT / "arrival_pre_t60_expanded_selected.joblib"
    joblib.dump(bundle, path)
    print({"saved": str(path)})

## Guardrails

- Lower validation error does not prove future reliability; report both tests.
- The new months improve coverage but remain non-consecutive snapshots.
- Weather stays deferred until this expanded flight-only baseline is frozen.
- CNN/LSTM remains secondary until dense continuous sequences are available.